# Fair FNO-family rerun (report Sec 9.27 follow-up)

Sec 9.27 found the raw-temperature FNOs *underfit*: the loss was dominated by the per-scenario offset and early stopping
ended training before the one-cycle schedule annealed. `scripts/fno_fair_cv.py` fixes both: the target is the rise over
ambient, the loss adds a detrended term, and early stopping is held off until `--min-epochs`. Three seeds, the notebook's
folds, early stopping on a hold-out drawn from the training fold. Results are written after every seed, and the script
resumes, so a session timeout loses at most one seed.

Attach `rajulkabir/thermo-3dic-shelf-v5` (geometry4/5/6 layout) and `rajulkabir/thermo-geometry7-pilot-v5` (geometry7,
fixed placement). Internet on, GPU T4.

**Part A.**

In [ ]:
import subprocess, sys, torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pyamg'], check=True)

In [ ]:
import shutil
from pathlib import Path
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
CLONE, DATA = Path('/kaggle/working/repo'), Path('/kaggle/working/data')
OUT = Path('/kaggle/working/fno_fair'); OUT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
for g, sub in [('geometry4', 'layout'), ('geometry5', 'layout'), ('geometry6', 'layout'), ('geometry7', 'pilot')]:
    files = sorted({p.name: p for p in Path('/kaggle/input').rglob(g + '_*.npz')}.values(), key=lambda p: p.name)
    dst = DATA / (f'3d-ice-layout-{g}' if sub == 'layout' else f'3d-ice-{g}-pilot') / g
    shutil.rmtree(dst, ignore_errors=True); dst.mkdir(parents=True)
    for p in files:
        (dst / p.name).symlink_to(p)
    print(g, len(files), 'files')

In [ ]:
import time
JOBS = [('geometry4', '3d-ice-layout-geometry4', ['fno', 'lt-fno'], [0, 1, 2]),
        ('geometry6', '3d-ice-layout-geometry6', ['fno', 'lt-fno'], [0, 1, 2]),
        ('geometry4', '3d-ice-layout-geometry4', ['cno-fno-attn'], [0])]
COMMON = ['--epochs', '300', '--min-epochs', '200', '--patience', '40', '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
for geom, ddir, variants, seeds in JOBS:
    out = OUT / f'{geom}.json'
    cmd = [sys.executable, 'scripts/fno_fair_cv.py', geom, '--data-dir', str(DATA / ddir), '--variants', *variants,
           '--seeds', *map(str, seeds), '--out', str(out)] + COMMON
    print('===', geom, variants, seeds, flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, cwd=CLONE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line and 'INFO' not in line and 'super().__init__' not in line and 'model = FNO3d' not in line:
            print('  ' + line, end='', flush=True)
    print(f'  exit {p.wait()} ({(time.time() - t) / 60:.0f} min)')

In [ ]:
import json, numpy as np
for f in sorted(OUT.glob('*.json')):
    r = json.loads(f.read_text())
    print(f.stem)
    for v in sorted({k.rsplit('_seed', 1)[0] for k in r}):
        runs = [r[k] for k in r if k.rsplit('_seed', 1)[0] == v]
        g = lambda key: f"{np.mean([x[key] for x in runs]):.3f}+-{np.std([x[key] for x in runs]):.3f}"
        print(f"  {v:<14} n_seeds {len(runs)}  R2 {g('r2_mean')}  det.MAE {g('det_mae_K')}  |peak| {g('abs_peak_err_K')}  "
              f"recall {g('recall_mean')}  loc {np.mean([x['loc_err_um_median'] for x in runs]):.0f}")